# 🤙 Reconhecimento de Gestos com Webcam
**Bibliotecas:** OpenCV + MediaPipe (Google)  
**Modelo:** MediaPipe Gesture Recognizer

Este notebook detecta gestos de mão em tempo real usando sua webcam.  
Gestos suportados: 👍 Thumbs Up, 👎 Thumbs Down, ✌️ Victory, ☝️ Pointing Up, 🤙 ILoveYou, ✊ Closed Fist, 🖐️ Open Palm.

## 📦 1. Instalação das dependências

Instale com `uv` antes de abrir o notebook:
```bash
uv add opencv-python mediapipe
```

Ou direto na célula:

In [ ]:
# Descomente se precisar instalar no ambiente do notebook
# !pip install opencv-python mediapipe -q

## 📥 2. Download do modelo

O MediaPipe Gesture Recognizer usa um modelo `.task` próprio para reconhecimento de gestos.

In [1]:
import urllib.request
import os

MODEL_URL  = "https://storage.googleapis.com/mediapipe-models/gesture_recognizer/gesture_recognizer/float16/1/gesture_recognizer.task"
MODEL_PATH = "gesture_recognizer.task"

if not os.path.exists(MODEL_PATH):
    print("Baixando modelo...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print(f"✅ Modelo salvo em: {MODEL_PATH}")
else:
    print(f"✅ Modelo já existe: {MODEL_PATH}")

Baixando modelo...
✅ Modelo salvo em: gesture_recognizer.task


## 📚 3. Importações

In [2]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
import numpy as np
import matplotlib.pyplot as plt

## ⚙️ 4. Configurações

In [3]:
# ── Índice da webcam (0 = câmera padrão) ─────────────────────────────────────
CAMERA_INDEX = 0

# ── Score mínimo de confiança para exibir um gesto (0.0 – 1.0) ───────────────
SCORE_THRESHOLD = 0.5

# ── Número máximo de mãos detectadas simultaneamente ─────────────────────────
MAX_HANDS = 2

# ── Tecla para encerrar (pressione durante a janela do OpenCV) ────────────────
TECLA_SAIR = "q"

# ── Emojis por gesto ─────────────────────────────────────────────────────────
EMOJI_GESTOS = {
    "Thumb_Up":      "👍",
    "Thumb_Down":    "👎",
    "Victory":       "✌️",
    "Pointing_Up":   "☝️",
    "ILoveYou":      "🤙",
    "Closed_Fist":   "✊",
    "Open_Palm":     "🖐️",
    "None":          "❓",
}

# ── Cores por mão (BGR) ───────────────────────────────────────────────────────
CORES_MAO = [
    (0, 255, 120),   # verde  — mão 1
    (255, 180, 0),   # laranja — mão 2
]

print("Configurações carregadas.")

Configurações carregadas.


## 🤖 5. Inicialização do Gesture Recognizer

In [4]:
base_options = mp_python.BaseOptions(model_asset_path=MODEL_PATH)

options = mp_vision.GestureRecognizerOptions(
    base_options=base_options,
    running_mode=mp_vision.RunningMode.IMAGE,
    num_hands=MAX_HANDS,
    min_hand_detection_confidence=SCORE_THRESHOLD,
    min_hand_presence_confidence=SCORE_THRESHOLD,
    min_tracking_confidence=SCORE_THRESHOLD,
)

recognizer = mp_vision.GestureRecognizer.create_from_options(options)
print("✅ Gesture Recognizer pronto!")

✅ Gesture Recognizer pronto!


## 🔍 6. Funções Auxiliares

In [9]:
def frame_para_mp(frame_bgr):
    """Converte frame BGR do OpenCV para MediaPipe Image."""
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    return mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)


# Conexões dos 21 landmarks da mão (definidas manualmente — sem mp.solutions)
HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),         # polegar
    (0,5),(5,6),(6,7),(7,8),         # indicador
    (0,9),(9,10),(10,11),(11,12),    # médio
    (0,13),(13,14),(14,15),(15,16),  # anelar
    (0,17),(17,18),(18,19),(19,20),  # mínimo
    (5,9),(9,13),(13,17),            # nós do meio
]


def desenhar_landmarks(frame, hand_landmarks_list):
    """Desenha os 21 landmarks de cada mão detectada."""
    h, w = frame.shape[:2]

    for idx, hand_landmarks in enumerate(hand_landmarks_list):
        cor    = CORES_MAO[idx % len(CORES_MAO)]
        pontos = [(int(lm.x * w), int(lm.y * h)) for lm in hand_landmarks]

        for inicio, fim in HAND_CONNECTIONS:
            cv2.line(frame, pontos[inicio], pontos[fim], cor, 2)

        for ponto in pontos:
            cv2.circle(frame, ponto, 5, cor, -1)
            cv2.circle(frame, ponto, 5, (255, 255, 255), 1)

    return frame

## 🚀 7. Loop Principal — Reconhecimento em Tempo Real

> **Para encerrar:** pressione `q` na janela do OpenCV.

In [14]:
cap = cv2.VideoCapture(CAMERA_INDEX)

if not cap.isOpened():
    print(f"❌ Não foi possível abrir a câmera (índice {CAMERA_INDEX}).")
    print("   Tente mudar CAMERA_INDEX para 1 ou 2 nas configurações.")
else:
    print("✅ Câmera aberta. Pressione 'q' na janela para encerrar.")

    while True:
        ret, frame = cap.read()

        if not ret:
            print("⚠️  Falha ao capturar frame.")
            break

        # Espelha o frame para ficar mais natural
        frame = cv2.flip(frame, 1)

        # Reconhecimento
        mp_image  = frame_para_mp(frame)
        resultado = recognizer.recognize(mp_image)

        # Desenha landmarks e gestos
        if resultado.hand_landmarks:
            frame = desenhar_landmarks(frame, resultado.hand_landmarks)

        if resultado.gestures:
            frame = desenhar_gestos(frame, resultado.gestures, resultado.handedness)
        else:
            cv2.putText(frame, "Nenhuma mao detectada", (10, 35),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (100, 100, 100), 2, cv2.LINE_AA)

        cv2.imshow("Reconhecimento de Gestos — pressione Q para sair", frame)

        if cv2.waitKey(1) & 0xFF == ord(TECLA_SAIR):
            print("Encerrando...")
            break

    cap.release()
    cv2.destroyAllWindows()
    print("✅ Câmera liberada.")

✅ Câmera aberta. Pressione 'q' na janela para encerrar.
Encerrando...
✅ Câmera liberada.


## 📸 8. Captura de Foto Única (opcional)

Reconhece gestos em um único frame e exibe o resultado inline no notebook.

In [ ]:
cap = cv2.VideoCapture(CAMERA_INDEX)
ret, frame = cap.read()
cap.release()

if ret:
    frame = cv2.flip(frame, 1)

    mp_image  = frame_para_mp(frame)
    resultado = recognizer.recognize(mp_image)

    frame_anotado = frame.copy()
    if resultado.hand_landmarks:
        frame_anotado = desenhar_landmarks(frame_anotado, resultado.hand_landmarks)
    if resultado.gestures:
        frame_anotado = desenhar_gestos(frame_anotado, resultado.gestures, resultado.handedness)

    # Exibe inline no notebook
    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(frame_anotado, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    n_maos = len(resultado.hand_landmarks) if resultado.hand_landmarks else 0
    plt.title(f"{n_maos} mão(s) detectada(s)", fontsize=14)
    plt.tight_layout()
    plt.show()

    # Lista gestos no terminal
    print("\nGestos detectados:")
    if resultado.gestures:
        for idx, (gesto_list, hand_list) in enumerate(zip(resultado.gestures, resultado.handedness)):
            gesto = gesto_list[0].category_name
            score = gesto_list[0].score
            mao   = hand_list[0].category_name
            emoji = EMOJI_GESTOS.get(gesto, "")
            print(f"  • Mão {idx + 1} ({mao}): {emoji} {gesto} — {score:.0%}")
    else:
        print("  Nenhum gesto detectado.")
else:
    print("❌ Não foi possível capturar o frame.")